## 5.0 Item Evaluation and Quality Control
### Two layers, in this order

A model can write 500 items in an hour. A content committee can review maybe 40 in a day. You close
that gap with two layers:

| Layer | What it is | Strength | Weakness |
|---|---|---|---|
| **1. Rules** | Plain Python checks | Free, instant, never changes its mind | Sees only the surface |
| **2. Model judge** | An LLM scoring against a rubric | Can read meaning | Fallible; worth nothing until calibrated |

> **Why rules first.** Moore et al. (2023) tested 200 items for 19 common item-writing flaws: a
> rule-based method caught **91%**, GPT-4 caught **79%**. Where a rule can be written, the rule wins
> and costs nothing. Spend the model only on what is left.


## Setup

In [ ]:
%pip install -q -U langchain-google-genai pydantic

In [ ]:
import os, re, time
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    print("Could not read GOOGLE_API_KEY from Colab Secrets.")

GEMINI_MODEL = "gemini-3.8-flash"     # if you get a 404, change this to a current Gemini model
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)


def ask(fn, *args):
    """Run a model call; retry twice if the service is busy, otherwise say what went wrong."""
    for attempt in range(3):
        try:
            return fn(*args)
        except Exception as e:
            busy = "503" in str(e) or "429" in str(e)
            if busy and attempt < 2:
                print("Model busy, retrying in 5 seconds...")
                time.sleep(5)
            else:
                print(f"Call failed: {type(e).__name__}: {str(e)[:120]}")
                return None


print("Ready. Model:", GEMINI_MODEL)

Ready. Model: gemini-3.8-flash


## The pool

Seven grade-school items. Read them before running anything; you can probably spot most of the
problems yourself. The goal is to make a machine agree with you.

| Item | Planted problem | Who should catch it |
|---|---|---|
| S01 | none | nobody |
| S02 | the key is not among the options | a rule |
| S03 | two options say the same thing | a rule |
| S04 | the key is much longer than the others | a rule (as a flag) |
| S06 | **the trap**: looks like a duplicate, is not | nobody; a rule must stay quiet |
| S07 | the key is simply wrong | only a reader |
| S08 | the stem asks nothing answerable | only a reader |

In [ ]:
class Item(BaseModel):
    item_id: str
    stem: str
    options: list[str]          # four options, in order A, B, C, D
    key: str                    # the letter of the correct option

    def key_text(self):
        i = "ABCD".find(self.key)
        return self.options[i] if 0 <= i < len(self.options) else None


POOL = [
    Item(item_id="S01", stem="Which gas do plants take in from the air during photosynthesis?",
         options=["Oxygen", "Carbon dioxide", "Nitrogen", "Hydrogen"], key="B"),
    Item(item_id="S02", stem="What is the boiling point of pure water at sea level, in degrees Celsius?",
         options=["0", "50", "100", "212"], key="E"),
    Item(item_id="S03", stem="A train travels 60 kilometres in 45 minutes. What is its average speed?",
         options=["80 km/h", "80 kilometres per hour", "45 km/h", "75 km/h"], key="A"),
    Item(item_id="S04", stem="Why does a metal spoon left in hot soup become warm?",
         options=["It melts", "Heat is conducted through the solid to the handle",
                  "It absorbs steam", "It reflects heat"], key="B"),
    Item(item_id="S06", stem="What is the value of x in the equation x^2 = 25, if x is negative?",
         options=["5", "-5", "25", "-25"], key="B"),
    Item(item_id="S07", stem="How many bones are in the adult human body?",
         options=["106", "206", "306", "406"], key="C"),
    Item(item_id="S08", stem="Which of the following is true about the water cycle?",
         options=["It is important for living things",
                  "Moisture moves between the sea, the air and the land",
                  "It involves clouds and rain in some way",
                  "It happens everywhere on the Earth's surface"], key="B"),
]

print(f"{len(POOL)} items in the pool.")

7 items in the pool.


---
## Layer 1: rule-based checks

Every check follows one contract:

```python
def my_check(item) -> str | None:   # a reason if something is wrong, None if not
```

Returning a **reason**, not `True`, means the check explains itself to the reviewer, can be counted
across a pool, can be tested, and gives the same answer every time.

We write three. The first two find items that are **broken**; the third finds an item that is only
**flawed**.

### Check 1: the key is not among the options

The model names a correct answer it did not put in the list. The item is not low quality; it is
broken. No reviewer should ever see it.

In [ ]:
def key_not_in_options(item):
    if item.key_text() is None:
        return f"key is '{item.key}' but there are only {len(item.options)} options"
    return None


print("S02:", key_not_in_options(POOL[1]))   # broken
print("S01:", key_not_in_options(POOL[0]))   # fine

S02: key is 'E' but there are only 4 options
S01: None


### Check 2: two options say the same thing

`"80 km/h"` and `"80 kilometres per hour"` are different strings, so we **normalise** first, and the
normalisation is the whole check:

- Normalise too little and you miss real duplicates (S03).
- Normalise too much and you destroy real distinctions: strip the minus sign and `-5` becomes `5`,
  and S06, a good item, gets rejected.

Keep an item like S06 in your test set forever.

In [ ]:
SYNONYMS = {   # your content conventions: a programme decision, so document it
    r"\bkilometres per hour\b": "km/h",
    r"\bkilometers per hour\b": "km/h",
    r"\bpercent\b": "%",
}


def normalise(text):
    t = text.lower().strip()
    for pattern, replacement in SYNONYMS.items():
        t = re.sub(pattern, replacement, t)
    return re.sub(r"\s+", " ", t)          # deliberately NOT stripping '-', '$', '%'


def duplicate_options(item):
    seen = {}
    for letter, text in zip("ABCD", item.options):
        n = normalise(text)
        if n in seen:
            return f"options {seen[n]} and {letter} are the same once normalised ('{n}')"
        seen[n] = letter
    return None


print("S03:", duplicate_options(POOL[2]))   # real duplicate: caught
print("S06:", duplicate_options(POOL[4]))   # the trap: must stay None

S03: options A and B are the same once normalised ('80 km/h')
S06: None


### Check 3: the key is much longer than the distractors

Test-wise students pick the long option. But S04 is usable; it just needs an edit. So this check
produces a **flag** for a human, not a rejection.

This is also the first check **with numbers in it**, and the numbers are yours to defend. Watch what
happens with only a ratio (`floor=0`), and then with a minimum length added.

In [ ]:
def key_is_longest(item, margin=1.6, floor=25):
    key = item.key_text()
    if key is None:
        return None                     # check 1 owns this failure
    others = [len(t) for l, t in zip("ABCD", item.options) if l != item.key]
    mean_other = sum(others) / len(others)
    if len(key) >= floor and len(key) > margin * mean_other:
        return f"key is {len(key)} chars vs a distractor mean of {mean_other:.0f}: length may cue it"
    return None


print("Ratio only (floor=0):")
for it in POOL:
    if key_is_longest(it, floor=0):
        print(" ", it.item_id, key_is_longest(it, floor=0))

print("\nRatio + minimum length (floor=25):")
for it in POOL:
    if key_is_longest(it):
        print(" ", it.item_id, key_is_longest(it))

Ratio only (floor=0):
  S01 key is 14 chars vs a distractor mean of 7: length may cue it
  S04 key is 49 chars vs a distractor mean of 13: length may cue it

Ratio + minimum length (floor=25):
  S04 key is 49 chars vs a distractor mean of 13: length may cue it


**With a ratio alone, S01 fires.** "Carbon dioxide" is twice as long as "Oxygen", but nobody picks it
because of that. A ratio has no sense of scale. Adding a minimum length fixes it, and now `margin`
and `floor` are two numbers you will be asked to justify.

> Choose them by running the check on items your reviewers have **already** judged and tuning until
> it agrees with them. That is a small calibration study, and it turns a number you chose into a
> number you can defend.

### Run all three: hard and soft

Nothing in the code makes a check hard or soft. **You** decide, and you can be asked why.

| Check | Severity | Meaning |
|---|---|---|
| `key_not_in_options` | **hard** | Not an item: rejected |
| `duplicate_options` | **hard** | No single defensible answer: rejected |
| `key_is_longest` | **soft** | Usable, but a student could score above their knowledge: a human decides |

In [ ]:
CHECKS = [
    ("key_not_in_options", "hard", key_not_in_options),
    ("duplicate_options",  "hard", duplicate_options),
    ("key_is_longest",     "soft", key_is_longest),
]


def run_checks(item):
    """Every check that fired, as (name, severity, reason)."""
    return [(name, sev, fn(item)) for name, sev, fn in CHECKS if fn(item)]


for it in POOL:
    flags = run_checks(it)
    print(f"{it.item_id}  " + ("clean" if not flags else
          "; ".join(f"{sev.upper()} {name}: {reason}" for name, sev, reason in flags)))

S01  clean
S02  HARD key_not_in_options: key is 'E' but there are only 4 options
S03  HARD duplicate_options: options A and B are the same once normalised ('80 km/h')
S04  SOFT key_is_longest: key is 49 chars vs a distractor mean of 13: length may cue it
S06  clean
S07  clean
S08  clean


---
## Layer 2: a model judge

The judge scores what a rule cannot see. Three design decisions do the work:

1. **The rubric is data, not code.** Reviewers can read it, argue with it, and edit it without
   touching the scoring function.
2. **Every score has a written anchor.** This matters most. Aydin et al. (2026): the same three
   raters reached κ = **0.117** on a holistic judgement (Bloom's level) but κ = **0.751** on a
   concrete one (is the stem negatively worded?). Fu et al. (2024) raised reliability on seven
   dimensions from α ≈ 0.18–0.56 to 0.43–0.80 by rewriting the guideline alone.
3. **Structured output.** The model returns an object that validates or fails, never loose text.

> **Test before a dimension goes in the rubric:** can you state a failure in one sentence without
> "appropriate", "clear", or "high-quality"? If not, neither the model nor your experts will score
> it reliably.

In [ ]:
RUBRIC = """key_correct: Is the keyed option the only correct answer?
  1 = The key is wrong, or another option is equally defensible.
  2 = The key is best, but another option could be argued.
  3 = The key is correct and no other option is defensible.
stem_answerable: Can the stem be answered without reading the options?
  1 = The stem only points at a topic; it asks no answerable question.
  2 = The stem asks a question but leaves out needed information.
  3 = A knowledgeable student could answer before seeing the options.
distractors_plausible: Would a student who does not know the answer be tempted?
  1 = No distractor is tempting; the key is obvious by elimination.
  2 = One distractor is tempting; the others are filler.
  3 = At least two distractors reflect a specific misunderstanding."""


class Score(BaseModel):
    name: str = Field(description="dimension name, exactly as in the rubric")
    score: int = Field(ge=1, le=3)
    reason: str = Field(description="one sentence quoting the part of the item that drove the score")


class Review(BaseModel):
    scores: list[Score]


judge_llm = llm.with_structured_output(Review)


def judge(item):
    options = "\n".join(f"{l}. {t}" for l, t in zip("ABCD", item.options))
    prompt = (f"Score this multiple-choice item 1, 2 or 3 on each rubric dimension, using the "
              f"anchors exactly as written.\n\nRUBRIC\n{RUBRIC}\n\n"
              f"ITEM\n{item.stem}\n{options}\nKeyed answer: {item.key}")
    return judge_llm.invoke(prompt)

### Judge only what survived the rules

An item that failed a hard check is already rejected; paying a model to score it is wasted money.
Watch S07 and S08, the two items the rules missed.

In [ ]:
survivors = [it for it in POOL if not any(sev == "hard" for _, sev, _ in run_checks(it))]
print(f"{len(POOL)} items in -> {len(survivors)} to the judge "
      f"({len(POOL) - len(survivors)} rejected by rules at no cost)\n")

results = {}
for it in survivors:
    r = ask(judge, it)
    if r:
        results[it.item_id] = r
        print(f"{it.item_id}  " + "  ".join(f"{s.name}={s.score}" for s in r.scores))
        for s in r.scores:
            if s.score == 1:
                print(f"       -> {s.name}: {s.reason}")

7 items in -> 5 to the judge (2 rejected by rules at no cost)

Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retry

### LLM judge will needs to be properly reviewed

**It agrees with itself whether or not it is right.** Norman et al. (2026), 21 judge models and
about 541,000 judgements: test-retest reliability above **0.95**, but agreement with humans of only
**κ = 0.376–0.511**. Yamauchi et al. (2025) removed the rubric from the prompt: validity fell from
0.666 to 0.487 while α stayed near 0.90. **Self-consistency is not evidence.**


In [ ]:
runs = [ask(judge, POOL[3]) for _ in range(3)]      # S04, a borderline item
runs = [r for r in runs if r]
for name in ["key_correct", "stem_answerable", "distractors_plausible"]:
    scores = [next((s.score for s in r.scores if s.name == name), None) for r in runs]
    print(f"{name:24s} {scores}")
print("\nStable is not the same as right. Only your reviewers can tell you if it is right.")

Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
Model busy, retrying in 5 seconds...
Model busy, retrying in 5 seconds...
Call failed: GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 
key_correct              []
stem_answerable          []
distractors_plausible    []

Stable is not the same as right. Only your reviewers can tell you if it is right.


---
## One gate, three words

Both layers feed one function that returns a **decision**, not a score: `pass`, `review`, or
`reject`. A number invites a meeting about the number; three words invite a meeting about the
**rule**, which is the thing you can write down and defend.

In [ ]:
def decide(item, review, min_score=2):
    flags = run_checks(item)
    reasons = [f"[{sev}] {name}: {reason}" for name, sev, reason in flags]
    if any(sev == "hard" for _, sev, _ in flags):
        return "reject", reasons
    if review:
        reasons += [f"[judge] {s.name}={s.score}: {s.reason}" for s in review.scores
                    if s.score < min_score]
    return ("review" if reasons else "pass"), reasons


counts = {"pass": 0, "review": 0, "reject": 0}
for it in POOL:
    status, reasons = decide(it, results.get(it.item_id))
    counts[status] += 1
    print(f"{it.item_id}  {status.upper()}")
    for r in reasons:
        print(f"        {r}")

print(f"\n{counts['pass']} pass, {counts['review']} review, {counts['reject']} reject. "
      f"Every item a reviewer sees arrives with its reason.")

S01  PASS
S02  REJECT
        [hard] key_not_in_options: key is 'E' but there are only 4 options
S03  REJECT
        [hard] duplicate_options: options A and B are the same once normalised ('80 km/h')
S04  REVIEW
        [soft] key_is_longest: key is 49 chars vs a distractor mean of 13: length may cue it
S06  PASS
S07  PASS
S08  PASS

4 pass, 1 review, 2 reject. Every item a reviewer sees arrives with its reason.


---
## Before using an LLM Judge

**Ensure proof that it agrees with your reviewers.** Until then, the judge's scores are awell-organised opinion. The calibration study:

1. Take 100 items your reviewers have already decided on.
2. Run the judge blind to those decisions.
3. Report **Cohen's κ**, not raw agreement (raw agreement overstated κ by 38.6 points on average in
   Norman et al.).
4. Check the **lean**: does it disagree mostly in one direction?
5. Set `min_score` by the cost of each error. A false reject loses a usable item; a false accept puts
   a flawed item on a form. They are not equal.
6. Your reviewers needs to be trained also and they must agree too.
### More checks worth adding

| Check | Catches |
|---|---|
| Repeated text in every option | Options that restate the stem ("There are eight planets that...") |
| Stem word only in the key | "Which **gas**...?" with only the key saying "Nitrogen **gas**" |
| Pool-level: key balance | Too many keys in one position |
| Pool-level: near-duplicates | Enemy items across the bank |
